# 纯torch代码实现vllm中默认参数下的Embedding全过程


In [1]:
import torch
import numpy as np

In [2]:
MODEL_NAME = "Qwen/Qwen3-Embedding-0.6B"

texts = ["Hello Word, a test sentence"]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. Tokenizer
从 tokenizer 得到 token id 序列

In [3]:
import json
import re

merges_file = "/Users/junzerg/.cache/huggingface/hub/models--Qwen--Qwen3-Embedding-0.6B/snapshots/c54f2e6e80b2d7b7de06f51cec4959f6b3e03418/merges.txt"
vocab_file = "/Users/junzerg/.cache/huggingface/hub/models--Qwen--Qwen3-Embedding-0.6B/snapshots/c54f2e6e80b2d7b7de06f51cec4959f6b3e03418/vocab.json"

# 可以加载tokenizer_config获取一下特殊token
bos_token = None

eos_token = '<|im_end|>'

unk_token = None

pad_token = '<|endoftext|>'

In [4]:
with open(vocab_file, "r", encoding="utf-8") as f:
    encoder = json.load(f)

vocab_size = len(encoder)
decoder = {v: k for k, v in encoder.items()}
decoder

{0: '!',
 1: '"',
 2: '#',
 3: '$',
 4: '%',
 5: '&',
 6: "'",
 7: '(',
 8: ')',
 9: '*',
 10: '+',
 11: ',',
 12: '-',
 13: '.',
 14: '/',
 15: '0',
 16: '1',
 17: '2',
 18: '3',
 19: '4',
 20: '5',
 21: '6',
 22: '7',
 23: '8',
 24: '9',
 25: ':',
 26: ';',
 27: '<',
 28: '=',
 29: '>',
 30: '?',
 31: '@',
 32: 'A',
 33: 'B',
 34: 'C',
 35: 'D',
 36: 'E',
 37: 'F',
 38: 'G',
 39: 'H',
 40: 'I',
 41: 'J',
 42: 'K',
 43: 'L',
 44: 'M',
 45: 'N',
 46: 'O',
 47: 'P',
 48: 'Q',
 49: 'R',
 50: 'S',
 51: 'T',
 52: 'U',
 53: 'V',
 54: 'W',
 55: 'X',
 56: 'Y',
 57: 'Z',
 58: '[',
 59: '\\',
 60: ']',
 61: '^',
 62: '_',
 63: '`',
 64: 'a',
 65: 'b',
 66: 'c',
 67: 'd',
 68: 'e',
 69: 'f',
 70: 'g',
 71: 'h',
 72: 'i',
 73: 'j',
 74: 'k',
 75: 'l',
 76: 'm',
 77: 'n',
 78: 'o',
 79: 'p',
 80: 'q',
 81: 'r',
 82: 's',
 83: 't',
 84: 'u',
 85: 'v',
 86: 'w',
 87: 'x',
 88: 'y',
 89: 'z',
 90: '{',
 91: '|',
 92: '}',
 93: '~',
 94: '¡',
 95: '¢',
 96: '£',
 97: '¤',
 98: '¥',
 99: '¦',
 100: '§'

In [5]:
def bytes_to_unicode():
    """
    目标：把 所有 256 个字节（0–255） 映射到 可打印的 Unicode 字符。
    可以把每个字节变成安全的 Unicode 字符串（避免分词器处理不了控制字符）。
    """
    bs = (
        list(range(ord("!"), ord("~") + 1)) + list(range(ord("¡"), ord("¬") + 1)) + list(range(ord("®"), ord("ÿ") + 1))
    )
    cs = bs[:]
    n = 0
    for b in range(2**8):
        if b not in bs:
            bs.append(b)
            cs.append(2**8 + n)
            n += 1
    cs = [chr(n) for n in cs]
    return dict(zip(bs, cs))

In [6]:
byte_encoder = bytes_to_unicode()
byte_encoder

{33: '!',
 34: '"',
 35: '#',
 36: '$',
 37: '%',
 38: '&',
 39: "'",
 40: '(',
 41: ')',
 42: '*',
 43: '+',
 44: ',',
 45: '-',
 46: '.',
 47: '/',
 48: '0',
 49: '1',
 50: '2',
 51: '3',
 52: '4',
 53: '5',
 54: '6',
 55: '7',
 56: '8',
 57: '9',
 58: ':',
 59: ';',
 60: '<',
 61: '=',
 62: '>',
 63: '?',
 64: '@',
 65: 'A',
 66: 'B',
 67: 'C',
 68: 'D',
 69: 'E',
 70: 'F',
 71: 'G',
 72: 'H',
 73: 'I',
 74: 'J',
 75: 'K',
 76: 'L',
 77: 'M',
 78: 'N',
 79: 'O',
 80: 'P',
 81: 'Q',
 82: 'R',
 83: 'S',
 84: 'T',
 85: 'U',
 86: 'V',
 87: 'W',
 88: 'X',
 89: 'Y',
 90: 'Z',
 91: '[',
 92: '\\',
 93: ']',
 94: '^',
 95: '_',
 96: '`',
 97: 'a',
 98: 'b',
 99: 'c',
 100: 'd',
 101: 'e',
 102: 'f',
 103: 'g',
 104: 'h',
 105: 'i',
 106: 'j',
 107: 'k',
 108: 'l',
 109: 'm',
 110: 'n',
 111: 'o',
 112: 'p',
 113: 'q',
 114: 'r',
 115: 's',
 116: 't',
 117: 'u',
 118: 'v',
 119: 'w',
 120: 'x',
 121: 'y',
 122: 'z',
 123: '{',
 124: '|',
 125: '}',
 126: '~',
 161: '¡',
 162: '¢',
 163: '£',

## 什么是 BPE

BPE 是一种子词分词方法，常用于 NLP 模型中。它的核心思想是：

从字符开始，把文本拆成最小单位（通常是单个字符）。

统计所有连续字符对的出现频率。

把出现频率最高的字符对合并成一个新符号。

重复步骤 2-3，直到达到预设的词汇表大小。

这样可以得到一个“子词词典”，既可以表示常见词，也可以通过组合表示生僻词。

## 为什么要做BPE
[BPE在大模型Embedding中的作用](./tests/why_bpe.md)

## 怎么做BPE
how_bpe.md
见下面的代码

In [7]:
bpe_merges = []
with open(merges_file, encoding="utf-8") as merges_handle:
    for i, line in enumerate(merges_handle):
        line = line.strip()
        if (i == 0 and line.startswith("#version:")) or not line:
            continue
        bpe_merges.append(tuple(line.split()))
bpe_merges

[('Ġ', 'Ġ'),
 ('ĠĠ', 'ĠĠ'),
 ('i', 'n'),
 ('Ġ', 't'),
 ('ĠĠĠĠ', 'ĠĠĠĠ'),
 ('e', 'r'),
 ('ĠĠ', 'Ġ'),
 ('o', 'n'),
 ('Ġ', 'a'),
 ('r', 'e'),
 ('a', 't'),
 ('s', 't'),
 ('e', 'n'),
 ('o', 'r'),
 ('Ġt', 'h'),
 ('Ċ', 'Ċ'),
 ('Ġ', 'c'),
 ('l', 'e'),
 ('Ġ', 's'),
 ('i', 't'),
 ('a', 'n'),
 ('a', 'r'),
 ('a', 'l'),
 ('Ġth', 'e'),
 (';', 'Ċ'),
 ('Ġ', 'p'),
 ('Ġ', 'f'),
 ('o', 'u'),
 ('Ġ', '='),
 ('i', 's'),
 ('ĠĠĠĠ', 'ĠĠĠ'),
 ('in', 'g'),
 ('e', 's'),
 ('Ġ', 'w'),
 ('i', 'on'),
 ('e', 'd'),
 ('i', 'c'),
 ('Ġ', 'b'),
 ('Ġ', 'd'),
 ('e', 't'),
 ('Ġ', 'm'),
 ('Ġ', 'o'),
 ('ĉ', 'ĉ'),
 ('r', 'o'),
 ('a', 's'),
 ('e', 'l'),
 ('c', 't'),
 ('n', 'd'),
 ('Ġ', 'in'),
 ('Ġ', 'h'),
 ('en', 't'),
 ('i', 'd'),
 ('Ġ', 'n'),
 ('a', 'm'),
 ('ĠĠĠĠĠĠĠĠ', 'ĠĠĠ'),
 ('Ġt', 'o'),
 ('Ġ', 're'),
 ('-', '-'),
 ('Ġ', '{'),
 ('Ġo', 'f'),
 ('o', 'm'),
 (')', ';Ċ'),
 ('i', 'm'),
 ('č', 'Ċ'),
 ('Ġ', '('),
 ('i', 'l'),
 ('/', '/'),
 ('Ġa', 'nd'),
 ('u', 'r'),
 ('s', 'e'),
 ('Ġ', 'l'),
 ('e', 'x'),
 ('Ġ', 'S'),
 ('a', 'd'),
 (

In [8]:
bpe_ranks = dict(zip(bpe_merges, range(len(bpe_merges))))
bpe_ranks

{('Ġ', 'Ġ'): 0,
 ('ĠĠ', 'ĠĠ'): 1,
 ('i', 'n'): 2,
 ('Ġ', 't'): 3,
 ('ĠĠĠĠ', 'ĠĠĠĠ'): 4,
 ('e', 'r'): 5,
 ('ĠĠ', 'Ġ'): 6,
 ('o', 'n'): 7,
 ('Ġ', 'a'): 8,
 ('r', 'e'): 9,
 ('a', 't'): 10,
 ('s', 't'): 11,
 ('e', 'n'): 12,
 ('o', 'r'): 13,
 ('Ġt', 'h'): 14,
 ('Ċ', 'Ċ'): 15,
 ('Ġ', 'c'): 16,
 ('l', 'e'): 17,
 ('Ġ', 's'): 18,
 ('i', 't'): 19,
 ('a', 'n'): 20,
 ('a', 'r'): 21,
 ('a', 'l'): 22,
 ('Ġth', 'e'): 23,
 (';', 'Ċ'): 24,
 ('Ġ', 'p'): 25,
 ('Ġ', 'f'): 26,
 ('o', 'u'): 27,
 ('Ġ', '='): 28,
 ('i', 's'): 29,
 ('ĠĠĠĠ', 'ĠĠĠ'): 30,
 ('in', 'g'): 31,
 ('e', 's'): 32,
 ('Ġ', 'w'): 33,
 ('i', 'on'): 34,
 ('e', 'd'): 35,
 ('i', 'c'): 36,
 ('Ġ', 'b'): 37,
 ('Ġ', 'd'): 38,
 ('e', 't'): 39,
 ('Ġ', 'm'): 40,
 ('Ġ', 'o'): 41,
 ('ĉ', 'ĉ'): 42,
 ('r', 'o'): 43,
 ('a', 's'): 44,
 ('e', 'l'): 45,
 ('c', 't'): 46,
 ('n', 'd'): 47,
 ('Ġ', 'in'): 48,
 ('Ġ', 'h'): 49,
 ('en', 't'): 50,
 ('i', 'd'): 51,
 ('Ġ', 'n'): 52,
 ('a', 'm'): 53,
 ('ĠĠĠĠĠĠĠĠ', 'ĠĠĠ'): 54,
 ('Ġt', 'o'): 55,
 ('Ġ', 're'): 56,
 ('-', '-

In [9]:
# 预分词
import regex
PRETOKENIZE_REGEX = r"""
(?i:'s|'t|'re|'ve|'m|'ll|'d)|[^\r\n\p{L}\p{N}]?\p{L}+|\p{N}| ?[^\s\p{L}\p{N}]+[\r\n]*|\s*[\r\n]+|\s+(?!\S)|\s+
"""

pat = regex.compile(PRETOKENIZE_REGEX)

In [10]:
cache = {}

In [11]:
def get_pairs(word):
    """
    Return set of symbol pairs in a word.

    Word is represented as tuple of symbols (symbols being variable-length strings).
    """
    pairs = set()
    prev_char = word[0]
    for char in word[1:]:
        pairs.add((prev_char, char))
        prev_char = char
    return pairs

In [12]:
def bpe(token):
    # 缓存检查,避免重复计算
    if token in cache:
        return cache[token]
    # 初始化元组，每个元素是单个字符
    word = tuple(token)
    # 所有相邻符号对
    pairs = get_pairs(word)

    if not pairs:
        return token

    while True:
        # 循环合并符号对
        # bpe_ranks 保存了 BPE 词汇表中符号对的优先级/顺序
        # 选择排名最小（优先级最高）的符号对进行合并
        bigram = min(pairs, key=lambda pair: bpe_ranks.get(pair, float("inf")))
        if bigram not in bpe_ranks:
            break
        # 找到 first + second 的位置，把它合并成一个新的符号
        first, second = bigram
        new_word = []
        i = 0
        while i < len(word):
            try:
                j = word.index(first, i)
            except ValueError:
                new_word.extend(word[i:])
                break
            else:
                new_word.extend(word[i:j])
                i = j

            if word[i] == first and i < len(word) - 1 and word[i + 1] == second:
                new_word.append(first + second)
                i += 2
            else:
                new_word.append(word[i])
                i += 1
        # 更新 word 和 pairs
        new_word = tuple(new_word)
        word = new_word

        # 没有可合并的符号对，或者 word 只剩一个符号
        if len(word) == 1:
            break
        else:
            pairs = get_pairs(word)
    # 返回 BPE token
    # 输出一个字符串，子词用空格分开
    word = " ".join(word)
    # 并存入缓存 cache 提高性能
    cache[token] = word
    return word

In [13]:
# BPE 分词流程的入口函数，它把一个文本字符串切分成最终的 BPE token（子词）序列。
def tokenize(text):
    """Tokenize a string."""
    bpe_tokens = []
    for token in regex.findall(pat, text):
        token = "".join(
            byte_encoder[b] for b in token.encode("utf-8")
        )  # Maps all our bytes to unicode strings, avoiding control tokens of the BPE (spaces in our case)
        bpe_tokens.extend(bpe_token for bpe_token in bpe(token).split(" "))
    return bpe_tokens

In [14]:
tokenize_result = tokenize("Hello Word, a test sentence")
tokenize_result

['Hello', 'ĠWord', ',', 'Ġa', 'Ġtest', 'Ġsentence']

In [15]:
token_ids = [encoder[token] for token in tokenize_result]
token_ids

[9707, 9322, 11, 264, 1273, 11652]

In [16]:
# padding_strategy = latest
# truncation_strategy = 'longest_first'
# max_length = 32768
# kwargs = {}
# add_special_tokens = True
# return_tensors = pt
[  9707,   9322,     11,    264,   1273,  11652, 151643]

[9707, 9322, 11, 264, 1273, 11652, 151643]